<a href="https://colab.research.google.com/github/HEU-Allen/gaotong/blob/main/day3/colab_gpu_benchmark.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1. 安装最新版（支持 NumPy 2.x + CUDA 12.x）
!pip install onnxruntime-gpu -q

# 2. 验证
import onnxruntime as ort
print("✅ 可用 Providers:", ort.get_available_providers())

✅ 可用 Providers: ['TensorrtExecutionProvider', 'CUDAExecutionProvider', 'AzureExecutionProvider', 'CPUExecutionProvider']


In [2]:
import onnxruntime as ort
import numpy as np
import time

def benchmark(path, provider, n=100):
    """统一测试函数"""
    print(f"\n🔧 正在测试 {provider}...")
    sess = ort.InferenceSession(path, providers=[provider])
    name = sess.get_inputs()[0].name
    dummy = np.random.randn(1, 3, 640, 640).astype(np.float32)

    # 预热（关键：TensorRT 首次构建引擎需要时间）
    print("   预热中...")
    for _ in range(10):
        sess.run(None, {name: dummy})

    # 正式计时
    print("   正式测试（100次）...")
    times = []
    for _ in range(n):
        t0 = time.perf_counter()
        sess.run(None, {name: dummy})
        times.append((time.perf_counter()-t0)*1000)

    mean = np.mean(times)
    fps = 1000 / mean
    return mean, fps

print("="*55)
print("📊 YOLOv10-N 多后端推理对比（Colab T4 GPU）")
print("="*55)

# 1. CPU 基线
cpu_lat, cpu_fps = benchmark('yolov10n.onnx', 'CPUExecutionProvider')
print(f"✅ CPU 后端    | 延迟: {cpu_lat:.2f}ms | FPS: {cpu_fps:.1f}")

# 2. CUDA 后端
cuda_lat, cuda_fps = benchmark('yolov10n.onnx', 'CUDAExecutionProvider')
print(f"✅ CUDA 后端   | 延迟: {cuda_lat:.2f}ms | FPS: {cuda_fps:.1f}")

# 3. TensorRT 后端（NVIDIA 专用优化，首次构建引擎较慢）
trt_lat, trt_fps = benchmark('yolov10n.onnx', 'TensorrtExecutionProvider')
print(f"✅ TensorRT    | 延迟: {trt_lat:.2f}ms | FPS: {trt_fps:.1f}")

print("="*55)
print("📈 加速对比（相对 CPU）:")
print(f"   CUDA 加速: {cpu_lat/cuda_lat:.2f}x")
print(f"   TensorRT 加速: {cpu_lat/trt_lat:.2f}x")
print("="*55)

📊 YOLOv10-N 多后端推理对比（Colab T4 GPU）

🔧 正在测试 CPUExecutionProvider...
   预热中...
   正式测试（100次）...
✅ CPU 后端    | 延迟: 156.45ms | FPS: 6.4

🔧 正在测试 CUDAExecutionProvider...
   预热中...
   正式测试（100次）...
✅ CUDA 后端   | 延迟: 136.67ms | FPS: 7.3

🔧 正在测试 TensorrtExecutionProvider...
*************** EP Error ***************
EP Error /onnxruntime_src/onnxruntime/python/onnxruntime_pybind_state.cc:456 void onnxruntime::python::RegisterTensorRTPluginsAsCustomOps(onnxruntime::python::PySessionOptions&, const ProviderOptions&) Please install TensorRT libraries as mentioned in the GPU requirements page, make sure they're in the PATH or LD_LIBRARY_PATH, and that your GPU is supported.
 when using ['TensorrtExecutionProvider']
Falling back to ['CPUExecutionProvider'] and retrying.
****************************************
   预热中...
   正式测试（100次）...
✅ TensorRT    | 延迟: 135.25ms | FPS: 7.4
📈 加速对比（相对 CPU）:
   CUDA 加速: 1.14x
   TensorRT 加速: 1.16x


In [3]:
# 1. 检查 GPU 型号
!nvidia-smi

# 2. 检查 PyTorch 能否用 GPU
import torch
print(f"PyTorch 版本: {torch.__version__}")
print(f"CUDA 可用: {torch.cuda.is_available()}")
print(f"GPU 数量: {torch.cuda.device_count()}")
if torch.cuda.is_available():
    print(f"GPU 名称: {torch.cuda.get_device_name(0)}")

Tue Jul  7 06:44:22 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
# 1. 安装 ultralytics
!pip install ultralytics -q

# 2. 加载模型并测试
from ultralytics import YOLO
import torch
import time
import numpy as np

model = YOLO('yolov10n.pt')

# ========== CPU 测试 ==========
print("🔧 CPU 推理测试...")
model.to('cpu')
dummy = torch.randn(1, 3, 640, 640)

for _ in range(10):
    model.predict(dummy, verbose=False)

times = []
for _ in range(100):
    t0 = time.perf_counter()
    model.predict(dummy, verbose=False)
    times.append((time.perf_counter()-t0)*1000)

cpu_mean = np.mean(times)
cpu_fps = 1000 / cpu_mean
print(f"✅ CPU | 延迟: {cpu_mean:.2f}ms | FPS: {cpu_fps:.1f}")

# ========== GPU 测试 ==========
print("\n🔧 GPU 推理测试...")
model.to('cuda')
dummy = dummy.to('cuda')

torch.cuda.synchronize()
for _ in range(10):
    model.predict(dummy, verbose=False)
torch.cuda.synchronize()

times = []
for _ in range(100):
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    model.predict(dummy, verbose=False)
    torch.cuda.synchronize()
    times.append((time.perf_counter()-t0)*1000)

gpu_mean = np.mean(times)
gpu_fps = 1000 / gpu_mean
print(f"✅ GPU | 延迟: {gpu_mean:.2f}ms | FPS: {gpu_fps:.1f}")

# ========== 结果对比 ==========
print("\n" + "="*50)
print("📊 PyTorch YOLOv10-N CPU vs GPU 对比")
print("="*50)
print(f"CPU (Colab) : {cpu_mean:.2f}ms | {cpu_fps:.1f} FPS")
print(f"GPU (T4)    : {gpu_mean:.2f}ms | {gpu_fps:.1f} FPS")
print(f"GPU 加速比  : {cpu_mean/gpu_mean:.2f}x")
print("="*50)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.8/41.8 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.7/16.7 MB 83.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.5.1 which is incompatible.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
🔧 CPU 推理测试...
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1.0 but max value is 5.201351165771484. Dividing input by 255.
WARNING ⚠️ torch.Tensor inputs should be normalized 0.0-1

In [7]:
# 1. 尝试安装 RKNN Toolkit2
!pip install rknn-toolkit2 -q

# 2. 验证安装
try:
    from rknn.api import RKNN
    print("✅ RKNN Toolkit2 安装成功")
except Exception as e:
    print(f"❌ 安装失败: {e}")
    print("提示：Colab Python 3.12 可能不兼容，需要转 Docker 方案")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 38.8/38.8 MB 29.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.4/55.4 kB 6.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 19.1/19.1 MB 95.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 12.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.6/294.6 kB 30.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 118.1/118.1 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 797.2/797.2 MB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 410.6/410.6 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 14.1/14.1 MB 103.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.7/23.7 MB 63.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 823.6/823.6 kB 40.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [8]:
import cv2
import numpy as np
import os

# 生成 10 张随机校准图片
os.makedirs('calib', exist_ok=True)
with open('calib.txt', 'w') as f:
    for i in range(10):
        img = np.random.randint(0, 255, (640, 640, 3), dtype=np.uint8)
        path = f'calib/{i}.jpg'
        cv2.imwrite(path, img)
        f.write(path + '\n')

print("✅ 校准数据集生成完成（10张 640x640 随机图片）")

✅ 校准数据集生成完成（10张 640x640 随机图片）


In [10]:
from rknn.api import RKNN

print("🔄 开始 ONNX → RKNN 转换...")

rknn = RKNN()

# 第一步：先配置（必须在 load_onnx 之前）
print("   配置 RK3588 优化参数...")
rknn.config(
    mean_values=[[0, 0, 0]],
    std_values=[[255, 255, 255]],
    target_platform='rk3588',
    optimization_level=3
)

# 第二步：加载 ONNX
print("   加载 ONNX 模型...")
rknn.load_onnx(model='yolov10n.onnx')

# 第三步：构建 INT8 量化
print("🔧 正在构建 INT8 量化模型（可能需要 1-2 分钟）...")
rknn.build(do_quantization=True, dataset='calib.txt')

# 第四步：导出
rknn.export_rknn('yolov10n.rknn')
print("✅ RKNN 模型导出完成：yolov10n.rknn")

rknn.release()

I rknn-toolkit2 version: 2.3.2


🔄 开始 ONNX → RKNN 转换...
   配置 RK3588 优化参数...
   加载 ONNX 模型...


I Loading : 100%|██████████████████████████████████████████████| 191/191 [00:00<00:00, 31465.52it/s]
E load_onnx: Traceback (most recent call last):
  File "rknn/api/rknn_log.py", line 344, in rknn.api.rknn_log.error_catch_decorator.error_catch_wrapper
  File "rknn/api/rknn_base.py", line 1579, in rknn.api.rknn_base.RKNNBase.load_onnx
  File "rknn/api/rknn_base.py", line 613, in rknn.api.rknn_base.RKNNBase._create_ir_and_inputs_meta
  File "rknn/api/ir_graph.py", line 84, in rknn.api.ir_graph.IRGraph.__init__
  File "rknn/api/ir_graph.py", line 665, in rknn.api.ir_graph.IRGraph.rebuild
  File "rknn/api/base_utils.py", line 34, in rknn.api.base_utils.to_np_type
AttributeError: module 'onnx' has no attribute 'mapping'

I ===================== WARN(0) =====================
E rknn-toolkit2 version: 2.3.2


ValueError: Traceback (most recent call last):
  File "rknn/api/rknn_log.py", line 344, in rknn.api.rknn_log.error_catch_decorator.error_catch_wrapper
  File "rknn/api/rknn_base.py", line 1579, in rknn.api.rknn_base.RKNNBase.load_onnx
  File "rknn/api/rknn_base.py", line 613, in rknn.api.rknn_base.RKNNBase._create_ir_and_inputs_meta
  File "rknn/api/ir_graph.py", line 84, in rknn.api.ir_graph.IRGraph.__init__
  File "rknn/api/ir_graph.py", line 665, in rknn.api.ir_graph.IRGraph.rebuild
  File "rknn/api/base_utils.py", line 34, in rknn.api.base_utils.to_np_type
AttributeError: module 'onnx' has no attribute 'mapping'


In [11]:
# 降级 onnx 到 RKNN 兼容版本
!pip install onnx==1.14.0 -q

# 验证
import onnx
print(f"✅ onnx 版本: {onnx.__version__}")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.3/11.3 MB 103.2 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done
  Preparing metadata (pyproject.toml) ... done
  error: subprocess-exited-with-error
  
  × Building wheel for onnx (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for onnx
ERROR: ERROR: Failed to build installable wheels for some pyproject.toml based projects (onnx)
✅ onnx 版本: 1.22.0


In [14]:
# 1. 接受 Conda Terms of Service（必须步骤）
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# 2. 创建 Python 3.9 环境
!conda create -n rknn python=3.9 -y

# 3. 安装 RKNN + 兼容的 onnx
!/usr/local/envs/rknn/bin/pip install rknn-toolkit2 onnx==1.14.0 -q

# 4. 验证
!/usr/local/envs/rknn/bin/python -c "from rknn.api import RKNN; import onnx; print('✅ RKNN 导入成功'); print('✅ onnx 版本:', onnx.__version__)"

accepted Terms of Service for https://repo.anaconda.com/pkgs/main
accepted Terms of Service for https://repo.anaconda.com/pkgs/r
Jupyter detected...
2 channel Terms of Service accepted
Retrieving notices: - \ | / - \ | / - \ | / - \ done
Channels:
 - defaults
Platform: linux-64
Solving environment: / done


==> WARNING: A newer version of conda exists. <==
    current version: 26.3.2
    latest version: 26.5.3

Please update conda by running

    $ conda update -n base -c defaults conda



## Package Plan ##

  environment location: /usr/local/envs/rknn

  added / updated specs:
    - python=3.9


The following packages will be downloaded:

    package                    |            build
    ---------------------------|-----------------
    _openmp_mutex-5.1          |           52_gnu           7 KB
    ca-certificates-2026.5.14  |       h06a4308_0         107 KB
    expat-2.8.2                |       h7354ed3_1          24 KB
    libexpat-2.8.2        

In [16]:
# 1. 把 RKNN 转换代码写到文件里
with open('/content/rknn_convert.py', 'w') as f:
    f.write('''from rknn.api import RKNN

print("🔄 开始 ONNX → RKNN 转换...")

rknn = RKNN()

# 配置
rknn.config(
    mean_values=[[0, 0, 0]],
    std_values=[[255, 255, 255]],
    target_platform='rk3588',
    optimization_level=3
)

# 加载 ONNX
rknn.load_onnx(model='yolov10n.onnx')

# 构建 INT8
print("🔧 构建 INT8 量化模型...")
rknn.build(do_quantization=True, dataset='calib.txt')

# 导出
rknn.export_rknn('yolov10n.rknn')
print("✅ RKNN 导出完成：yolov10n.rknn")

rknn.release()
''')

# 2. 用 conda 的 Python 3.9 执行
!/usr/local/envs/rknn/bin/python /content/rknn_convert.py

🔄 开始 ONNX → RKNN 转换...
/usr/local/envs/rknn/lib/python3.9/site-packages/rknn/api/rknn.py:51: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  self.rknn_base = RKNNBase(cur_path, verbose)
I rknn-toolkit2 version: 2.3.0

🔧 构建 INT8 量化模型...







W build: found outlier value, this may affect quantization accuracy
                        const name                                  abs_mean    abs_std     outlier value
                        model.23.one2one_cv3.1.1.1.conv.weight      0.50        0.61        -11.375     
                        model.23.one2one_cv3.0.1.1.conv.weight      0.61        0.62        -11.948     


W build: The default input dtype of 'images' is changed from 'float32' to 'int8' in rknn model for performance!
                       Please take care of this change when dep

In [18]:
# 1. 重新转换（target_platform 改为 rk3566，模拟器支持）
with open('/content/rknn_convert2.py', 'w') as f:
    f.write('''from rknn.api import RKNN

print("🔄 重新转换 ONNX → RKNN（rk3566 平台）...")

rknn = RKNN()

rknn.config(
    mean_values=[[0, 0, 0]],
    std_values=[[255, 255, 255]],
    target_platform='rk3566',  # 改为模拟器支持的平台
    optimization_level=3
)

rknn.load_onnx(model='yolov10n.onnx')
rknn.build(do_quantization=True, dataset='calib.txt')
rknn.export_rknn('yolov10n_rk3566.rknn')
print("✅ RKNN 导出完成：yolov10n_rk3566.rknn")

rknn.release()
''')

!/usr/local/envs/rknn/bin/python /content/rknn_convert2.py

# 2. 模拟器测试
with open('/content/rknn_test2.py', 'w') as f:
    f.write('''from rknn.api import RKNN
import numpy as np
import time

print("🔧 启动 RKNN 模拟器（rk3566）...")

rknn = RKNN()
rknn.load_rknn('yolov10n_rk3566.rknn')
rknn.init_runtime(target='simulator')

dummy = np.random.randint(0, 255, (1, 640, 640, 3), dtype=np.uint8)

# 预热
for _ in range(5):
    rknn.inference(inputs=[dummy])

# 计时
times = []
for _ in range(50):
    t0 = time.perf_counter()
    rknn.inference(inputs=[dummy])
    times.append((time.perf_counter()-t0)*1000)

mean_ms = np.mean(times)
fps = 1000 / mean_ms

print("\\n" + "="*45)
print("📊 RKNN 模拟器结果（模拟 rk3566 NPU）")
print("="*45)
print(f"   平均延迟 : {mean_ms:.2f} ms")
print(f"   理论 FPS : {fps:.1f}")
print("="*45)

rknn.release()
''')

!/usr/local/envs/rknn/bin/python /content/rknn_test2.py

🔄 重新转换 ONNX → RKNN（rk3566 平台）...
/usr/local/envs/rknn/lib/python3.9/site-packages/rknn/api/rknn.py:51: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  self.rknn_base = RKNNBase(cur_path, verbose)
I rknn-toolkit2 version: 2.3.0








W build: found outlier value, this may affect quantization accuracy
                        const name                                  abs_mean    abs_std     outlier value
                        model.23.one2one_cv3.1.1.1.conv.weight      0.50        0.61        -11.375     
                        model.23.one2one_cv3.0.1.1.conv.weight      0.61        0.62        -11.948     


W build: The default input dtype of 'images' is changed from 'float32' to 'int8' in rknn model for performance!
                       Please take care of this change when deploy rknn